# Детекция аномалий

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные и постановка задачи
2. Простые методы: z-score и правило Тьюки — и где они ломаются
3. Isolation Forest
4. LOF, One-Class SVM и сравнение методов
5. Практикум

Это занятие продолжает линию обучения без учителя. В кластеризации мы искали **группы похожих** объектов, а теперь ищем **редкие нетипичные** объекты — аномалии: сбои оборудования, мошеннические операции, ошибки в данных. Разметки почти нет (аномалий мало и они редко размечены), поэтому детектор строят по «норме», а не по классам.

Демонстрационный пример — телеметрия оборудования с двумя режимами нормальной работы и редкими сбоями. Практикум (раздел 5) вы выполняете на **датасете своего проекта**. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

Среда та же, что и в прошлых занятиях. **Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score

import wandb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)
RANDOM_STATE = 7
wandb.login()

## 1. Данные и постановка задачи

Оборудование работает в двух режимах, у каждого — свой «нормальный» разброс показаний датчиков (два плотных облака). Иногда случаются сбои: часть из них — **грубые выбросы** (датчик далеко ушёл от нормы), часть — **локальные аномалии** (значения сами по себе не крайние, но не похожи ни на один из режимов).

**Постановка.** Разметки у нас нет: считается известной только «норма». Нужно для каждого объекта получить **балл аномальности** и по порогу решать, отправлять ли объект на проверку. Истинные метки мы оставим только для оценки качества в конце.

*Обещание к оценке:* если аномалии — редкое событие, обычные метрики классификации (accuracy) бесполезны; работаем с ROC-AUC и **Average Precision** (связь с занятием 3).

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# два режима нормальной работы
mode_a = rng.normal([-3, -3], 0.8, size=(300, 2))
mode_b = rng.normal([3, 3], 0.8, size=(300, 2))
X_normal = np.vstack([mode_a, mode_b])

# сбои: 10 «грубых» (далеко) + 20 «локальных» (в зазоре между режимами)
far = rng.uniform(-10, 10, size=(15, 2))
far = far[np.abs(far).max(axis=1) > 5][:10]
local = rng.normal([0, 0], 0.7, size=(20, 2))
X_anom = np.vstack([far, local])

X = np.vstack([X_normal, X_anom])
y = np.r_[np.zeros(len(X_normal)), np.ones(len(X_anom))]

print("Объектов:", len(X), "| аномалий:", int(y.sum()), f"({y.mean() * 100:.1f}%)")

plt.figure(figsize=(8, 8))
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=20, alpha=0.5, label='норма')
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=40, color='red', label='аномалии (истина)')
plt.xlabel('датчик 1'); plt.ylabel('датчик 2')
plt.title('Телеметрия: два режима и редкие сбои')
plt.legend(); plt.show()

**Примечание.** В реальной задаче метки `y` неизвестны — их не используют при обучении, только для проверки метода. Именно поэтому детекция аномалий — задача **без учителя**.

*Жёлтая метрика:* у сигнала аномальности мы оцениваем **ранжирование** объектов (ROC-AUC) и качество на редком положительном классе (**Average Precision**).

## 2. Простые методы: z-score и правило Тьюки

### Математический аппарат

Самый простой способ — считать объект аномальным, если он далеко от центра нормы. По каждому признаку считают **z-оценку**:

$$z = \frac{x - \mu}{\sigma}$$

и объявляют аномалиями точки с $|z| > 3$ (эмпирическое правило «трёх сигм» для нормального распределения). Алгоритм-аналог, устойчивый к выбросам, — **правило Тьюки** по межквартильному размаху: границы $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$.

*Смысл:* эти методы «глобальные» — они опираются на положение относительно **всего** облака данных по каждой оси. Они ловят грубые выбросы, но **слепы к локальным аномалиям** (значение не крайнее, но не похоже ни на одну группу) и могут ошибаться, когда норма многомодальна.

In [ ]:
def report(name, score):
    print(f"{name:16s} ROC-AUC={roc_auc_score(y, score):.3f}  "
          f"AP={average_precision_score(y, score):.3f}")

# z-score по каждой оси: берём максимальную по модулю
z = np.abs((X - X.mean(axis=0)) / X.std(axis=0)).max(axis=1)
report("z-score", z)

# правило Тьюки: число осей, где объект вышел за границы
q1, q3 = np.percentile(X, [25, 75], axis=0)
iqr = q3 - q1
tukey = ((X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)).sum(axis=1).astype(float)
report("IQR (Тьюки)", tukey)

**Задача 1.**
Посмотрите на метрики z-score и правила Тьюки. Они оказались близки к случайному уровню (а z-score даже хуже). Объясните почему: что именно эти методы считают «аномальным» на наших данных и почему локальные аномалии в зазоре между режимами они пропускают? Ответ — в «Выводе».

**Вывод:**

_..._

## 3. Isolation Forest

### Математический аппарат

Идея **Isolation Forest** — не описывать норму, а оценивать, насколько **легко изолировать** объект. Лес случайно выбирает признак и порог и делит данные; аномалии (которые одиноки и отличаются) отсекаются за малое число разбиений, а типичные точки требуют много шагов. Балл аномальности:

$$s(x, n) = 2^{-\,\mathbb{E}[h(x)] / c(n)}$$

где $h(x)$ — глубина, на которой объект изолирован, $c(n)$ — нормировка на среднюю глубину для $n$ объектов. Чем ближе $s$ к 1, тем вероятнее аномалия.

*Смысл:* метод не требует разметки, не предполагает нормальности и хорошо работает с многомодальной нормой; гиперпараметр `contamination` задаёт ожидаемую долю аномалий (влияет только на порог и на `predict`, но не на сами баллы `score_samples`).

In [ ]:
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=RANDOM_STATE).fit(X)
iso_score = -iso.score_samples(X)   # больше — аномальнее
report("IsolationForest", iso_score)

# как размечает при ожидаемой доле 5%
pred = iso.predict(X)  # -1 — аномалия
tp = int(((pred == -1) & (y == 1)).sum())
fp = int(((pred == -1) & (y == 0)).sum())
fn = int(((pred == 1) & (y == 1)).sum())
print(f"При contamination=0.05: TP={tp}, FP={fp}, FN={fn}")

# визуализация: истинные аномалии и найденные
plt.figure(figsize=(8, 8))
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=20, alpha=0.4, label='норма')
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=60, facecolors='none', edgecolors='red', label='истина')
plt.scatter(X[pred == -1, 0], X[pred == -1, 1], s=120, marker='x', color='black', label='найдено')
plt.xlabel('датчик 1'); plt.ylabel('датчик 2')
plt.title('Isolation Forest: найденные аномалии')
plt.legend(); plt.show()

**Задача 2.**
Сколько аномалий нашёл Isolation Forest при `contamination=0.05` и сколько из них верные (TP/FP/FN)? Что изменится в разметке, если поставить `contamination=0.02` или `0.15`, и что при этом произойдёт с баллами `score_samples`? Ответ — в «Выводе».

**Вывод:**

_..._

## 4. LOF, One-Class SVM и сравнение методов

### Математический аппарат

- **Local Outlier Factor (LOF)** сравнивает локальную плотность объекта с плотностью его соседей: $LOF \gg 1$ означает, что объект находится в разреженной области рядом с плотными соседями (локальная аномалия). Метод хорошо ловит аномалии **внутри** данных, но чувствителен к числу соседей `n_neighbors`.
- **One-Class SVM** строит границу вокруг «нормы» в признаковом пространстве (обычно с RBF-ядром) и объявляет всё снаружи аномалией. Параметр `nu` задаёт приблизительно долю выбросов; метод чувствителен к масштабу признаков, поэтому данные нужно стандартизовать.

**Сравнение.** У методов разные представления об аномальности: глобальное отклонение (z-score, IQR), изоляция (Isolation Forest), локальная плотность (LOF), граница нормы (One-Class SVM). Единого «лучшего» нет — выбирают на валидации или по предметному смыслу.

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit(X)
lof_score = -lof.negative_outlier_factor_
report("LOF", lof_score)

ocsvm = OneClassSVM(nu=0.05, gamma='scale').fit(X)
ocsvm_score = -ocsvm.score_samples(X)
report("One-Class SVM", ocsvm_score)

print("\nСводка (чем выше метрики, тем лучше):")
summary = pd.DataFrame([
    ('z-score', roc_auc_score(y, z), average_precision_score(y, z)),
    ('IQR', roc_auc_score(y, tukey), average_precision_score(y, tukey)),
    ('IsolationForest', roc_auc_score(y, iso_score), average_precision_score(y, iso_score)),
    ('LOF', roc_auc_score(y, lof_score), average_precision_score(y, lof_score)),
    ('One-Class SVM', roc_auc_score(y, ocsvm_score), average_precision_score(y, ocsvm_score)),
], columns=['метод', 'ROC-AUC', 'AP'])
print(summary.round(3).to_string(index=False))

run = wandb.init(project="apml-course", name="anomaly-demo",
                 config={"contamination": 0.05, "n_neighbors": 20})
wandb.log({"if_roc_auc": roc_auc_score(y, iso_score), "if_ap": average_precision_score(y, iso_score)})
run.finish()

**Задача 3.**
Сравните методы по ROC-AUC и AP. Какой метод лучше всех, а какой почти бесполезен на этих данных и почему? Обратите внимание: у z-score ROC-AUC ниже 0.5 — что это означает на практике? Ответ — в «Выводе».

**Вывод:**

_..._

**Задача со звёздочкой.**
В задаче с очень редкими аномалиями (например, 1% объектов) ROC-AUC может выглядеть прилично, тогда как реальная работа с положительным классом — плохой. Объясните, почему в таких задачах предпочитают **Average Precision (PR-AUC)**, а не ROC-AUC (подсказка — та же логика дисбаланса, что в занятии 3).

**Ваш ответ:**

_..._

## 5. Практикум

Задания выполняются на **датасете своего проекта**. Детекцию аномалий удобно трактовать как «поиск подозрительных объектов»: ошибок ввода, нетипичных клиентов, редких сбоев. После каждого задания заполните ячейку **«Вывод»**.

### Задание 5.1. Признаки и постановка

Выберите набор числовых признаков для поиска аномалий, обработайте пропуски и стандартизуйте данные (методы на расстояниях к масштабу чувствительны). Сформулируйте, что именно вы считаете «аномалией» в предметной области.

**Вывод:**

_..._

### Задание 5.2. Простые методы

Найдите аномалии простыми правилами (z-score с порогом 3 и правило Тьюки). Сколько объектов помечено? Как выглядят эти объекты — действительно ли они странные?

**Вывод:**

_..._

### Задание 5.3. Isolation Forest и LOF

Примените Isolation Forest и LOF (`contamination` задайте исходя из ожидаемой доли аномалий в вашей задаче). Сравните найденные объекты между методами — много ли общих?

**Вывод:**

_..._

### Задание 5.4. Визуализация

Снизьте данные до двух измерений (PCA) и отметьте на графике найденные аномалии. Выглядят ли они как отдельные точки или как отдельные маленькие группы?

**Вывод:**

_..._

### Задание 5.5. Итоговое резюме

В 5–7 предложениях: какой метод вы считаете основным для своей задачи и почему; как задали порог/долю аномалий; что оказалось аномальным по смыслу; какие ограничения детектора важно указать (мало разметки, дрейф данных, цена ложных срабатываний).

**Резюме:**

_..._

---

## P.S. Автоэнкодеры как следующий шаг

Ещё один популярный подход к детекции аномалий — **автоэнкодер**: нейросеть, которая сжимает объект в короткое представление и восстанавливает обратно, обучаясь преимущественно на **норме**. Хорошо выученная норма восстанавливается точно, а нетипичный объект — плохо. Мерой аномальности становится **ошибка восстановления**.

Это прямое развитие идеи из второго занятия: **остаток модели на норме = сигнал аномальности**. Только теперь модель нелинейная — автоэнкодер. Подробно мы разберём его в блоке про нейросети (занятие 9).

### Справочные материалы

- Детекция новизны и выбросов (scikit-learn): https://scikit-learn.org/stable/modules/outlier_detection.html
- Isolation Forest: https://scikit-learn.org/stable/modules/outlier_detection.html#isolation-forest
- Local Outlier Factor: https://scikit-learn.org/stable/modules/outlier_detection.html#local-outlier-factor
- One-Class SVM: https://scikit-learn.org/stable/modules/outlier_detection.html#novelty-detection-with-a-svm
- Среднее против медианы (интерактив): https://www.reneshbedre.com/blog/mean-vs-median.html